# Alpha-Mind Quick Start 5: Five-Quantile Alpha Factor Analysis

In [ ]:
# %matplotlib inline
import asyncio
import datetime as dt


## 1. Data Preparation
------------------

* Universe: CSI 800
* Factor: `ETOP`
* Rebalancing frequency: five trading days
* Start date: 2010-01-01
* End date: 2019-01-31
* Number of quantiles: five

In [ ]:
# Factor-neutralization settings
# industry_stryles contains industry styles; risk_styles contains risk styles
# makeSchedule generates the business-day rebalancing sequence
import numpy as np
import pyarrow as pa


In [ ]:
# Fetch data
# fetch_factor_range fetches factor data
# fetch_risk_model_range fetches risk-model data
# fetch_dx_return_range fetches return data
#%%time
import pyarrow.compute as pc

rng = np.random.default_rng(42)


## 2. Comparing Quantile-Group Returns
-----------------------

In [ ]:
# neutralize is the factor-neutralization function
n_dates = 6
n_assets = 12


In [ ]:
# er_quantile_analysis is the quantile-analysis function
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.analysis.quantileanalysis import quantile_analysis


In [ ]:
latest = panel.filter(pc.equal(panel["trade_date"], dates[-1]))
returns = quantile_analysis(
    latest.select(["factor"]),
    np.array([1.0]),
    latest["dx"].to_numpy(zero_copy_only=False),
    n_bins=5,
    pre_process=[],
    post_process=[],
)


In [ ]:
assert returns.shape == (5,)
pa.table({"quantile": np.arange(1, 6), "return": returns})
